# MVCLTrainingSleepEEG on Sleep-EDF

This example shows how to use `MVCLTrainingSleepEEG` from `pyhealth.tasks`.

The workflow mirrors the task pattern used in PyHealth examples:
1. Load `SleepEDFDataset`.
2. Run the task on one patient for a quick sanity check.
3. Optionally run `set_task()` for the full dataset.
4. Run the pretraining step on the subset of SleepEDF data.
5. Save the model state.

In [ ]:
!uv pip install ipywidgets

import os

from pyhealth.datasets import SleepEDFDataset
from pyhealth.tasks import MVCLTrainingSleepEEG

# Update this absolute path to your local Sleep-EDF root.
DATA_ROOT = "C:\\Users\\shart\\workspace\\CS-598\\PyHealth\\sleepedf"
assert os.path.exists(DATA_ROOT), f"Sleep-EDF root path {DATA_ROOT} does not exist. Please update the path to your local Sleep-EDF root."
assert os.path.isabs(DATA_ROOT), f"Sleep-EDF root path {DATA_ROOT} is not an absolute path. Please update to an absolute path."

In [ ]:
dataset = SleepEDFDataset(root=DATA_ROOT, subset="cassette", dev=True)
dataset.stats()
print(f"Number of patients: {len(dataset.unique_patient_ids)}")

In [ ]:
"""
This dataset contains 153 whole-night sleep electroencephalography
(EEG) recordings collected from 82 healthy subjects. Each recording is sampled at 100 Hz using a 1-lead 
EEG signal. The EEG signals are segmented into non-overlapping windows of size 200, each forming
one sample. Each sample is labeled with one of five sleep stages: Wake (W), Non-rapid Eye Movement
(N1, N2, N3), and Rapid Eye Movement (REM). This segmentation results in 371,055 samples
"""

In [ ]:
# Quick sanity check on one patient.
patient_id = dataset.unique_patient_ids[0]
patient = dataset.get_patient(patient_id)

task = MVCLTrainingSleepEEG(
    window_size=200, ## Create None overlapping window of 200 lenth 
    crop_length=178, ## take first 178 data points of the window to match that of Epilepsy data 
    eeg_channel="EEG Fpz-Cz",
    root_path=DATA_ROOT, ## Pass the root path to the task so it can load the data correctly.
)
samples = task(patient)

print(f"patient_id: {patient_id}")
print(f"sample count: {len(samples)}")
print(f"sample keys: {list(samples[0].keys())}")
print(f"signal shape: {samples[0]['signal'].shape}")
print(f"xt shape: {samples[0]['xt'].shape}")
print(f"xd shape: {samples[0]['xd'].shape}")
print(f"xf shape: {samples[0]['xf'].shape}")
print(f"label: {samples[0]['label']}")

In [ ]:
# Full pipeline (can take a while and uses disk cache).
sample_dataset = dataset.set_task(task, num_workers=16)
print(f"Total task samples: {len(sample_dataset)}")
print(f"Input schema: {sample_dataset.input_schema}")
print(f"Output schema: {sample_dataset.output_schema}")

In [ ]:
# Factory functions and helpers required to setup the model.
import math
from pathlib import Path
from typing import Any, Dict, Dict, List, List, Mapping, Union

import torch
import torch.nn as nn

def augment_time(x: torch.Tensor, std: float = 0.1) -> torch.Tensor:
    """Time-domain jitter augmentation"""
    noise = torch.randn_like(x) * std
    return x + noise
    
def augment_freq(sample: torch.Tensor, pertub_ratio: float = 0.05) -> torch.Tensor:
    """Frequency-domain augmentation (remove and add frequencies)"""
    aug_1 = remove_frequency(sample, pertub_ratio)
    aug_2 = add_frequency(sample, pertub_ratio)
    return aug_1 + aug_2

def remove_frequency(x: torch.Tensor, pertub_ratio: float = 0.0) -> torch.Tensor:
    mask = torch.rand(x.shape, device=x.device) > pertub_ratio
    return x * mask

def add_frequency(x: torch.Tensor, pertub_ratio: float = 0.0) -> torch.Tensor:
    mask = torch.rand(x.shape, device=x.device) > (1 - pertub_ratio)
    max_amplitude = x.max()
    random_am = torch.rand(mask.shape, device=x.device) * (max_amplitude * 0.1)
    pertub_matrix = mask * random_am
    return x + pertub_matrix

class PositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1, max_len=1024):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, hidden_dim)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2) * 
                             (-math.log(10000.0) / hidden_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: [1, max_len, hidden_dim]
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
# Now we set up the model and training loop.
from pyhealth.datasets.sample_dataset import SampleDataset
from pyhealth.datasets.splitter import split_by_sample
from pyhealth.datasets.utils import get_dataloader
from pyhealth.models.mvcl_model import MultiViewContrastiveModel
from torch.optim import Adam
from torch.utils.data import DataLoader
from pyhealth.trainer import Trainer

pretrain_ds, _, _ = split_by_sample(sample_dataset, [0.05, 0.05, 0.9])
assert type(pretrain_ds) == SampleDataset, "Expected a SampleDataset after splitting"
pretrain_loader = get_dataloader(pretrain_ds, batch_size=128, shuffle=True)


view_keys=["xt", "xf", "xd"]
model = MultiViewContrastiveModel(
    dataset=pretrain_ds,
    encoders=torch.nn.ModuleDict({
        k: nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True),
            num_layers=2
        ) for k in view_keys
    }),
    projectors = nn.ModuleDict({
        k: nn.Linear(1, 128) 
        for k in view_keys
    }),
    augmentations={"xt": augment_time, "xf": augment_freq, "xd": augment_time},
    pos_encoders=nn.ModuleDict({
        k: PositionalEncoding(hidden_dim=128, dropout=0.1) for k in view_keys
    }),
    hidden_dim=128,
    training_stage="pretrain",
    num_classes=3

)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pretrainer = Trainer(
    model=model,
    device=str(device),
)
pretrainer.train(
    train_dataloader=pretrain_loader,
    epochs=2,
    optimizer_class=torch.optim.Adam,
    optimizer_params={
        "lr": 0.002,
        "weight_decay": 1e-5,
    },
)

In [ ]:
# Save the pretraining model state for later use.
import os
import tempfile

tmp_path = tempfile.mkdtemp()
torch.save(model.state_dict(), os.path.join(tmp_path, "mvcl_pretrain.pth"))